# Асинхронность в Python

Объект Корутина - функции, которые могут быть приостановлены и возобновлены

In [ ]:
# Пример работы корутины в Python с использованием asyncio

import asyncio

# НЕ ПРАВИЛЬНО
async def example_coroutine():
    print("Hello from coroutine!")

example_coroutine()  # создан объект корутины example_coroutine(), корутина не запущена!

# ПРАВИЛЬНО
async def example_coroutine():
    print("Hello from coroutine!")

# в Jupyter Notebook можно использовать await, в обычном скрипте нужно использовать asyncio.run() для запуска корутины
await example_coroutine()  # корутина example_coroutine() запущена

Объек Task - представляют собой обертки для корутин и используются для их конкурентного выполнения

In [ ]:
# Пример создания и запуска задачи

import asyncio

async def example_coroutine():
    await asyncio.sleep(1)
    print("Hello from coroutine!")

async def main():
    task = asyncio.create_task(example_coroutine())  # создаем задачу из корутины example_coroutine()
    await task # ждем выполнения задачи

# в Jupyter Notebook можно использовать await, в обычном скрипте нужно использовать asyncio.run() для запуска корутины
await main()

In [ ]:
# Пример создания и запуска нескольких задач

import asyncio

async def example_coroutine():
    await asyncio.sleep(1)
    print("Hello from coroutine!")

async def main():
    tasks = []
    for _ in range(10):
        task = asyncio.create_task(example_coroutine())  # создаем 10 задач
        tasks.append(task)  # добавляем все задачи в список tasks
    await asyncio.gather(*tasks)  # запускаем все задачи из списка tasks

# в Jupyter Notebook можно использовать await, в обычном скрипте нужно использовать asyncio.run() для запуска корутины
await main()

Future - результат асинхронной операции, который появится в будущем

Объект Future/Task может находиться в одном из трёх состояний:
- Pending (ожидание) -  в этом состоянии операция, связанная с объектом Future , еще не выполнена
- Completed (завершён) - когда операция завершена, объект Future переходит в состояние "завершено"
- Cancelled (отменён) - если операция была отменена, объект Future переходит в состояние "отменено"

In [ ]:
# Пример использования Future для получения результата из корутины

import asyncio


async def simulate_long_running_task(name, delay, future: asyncio.Future):
    print(f"Задача '{name}' началась, будет выполнена за {delay} секунд.")
    await asyncio.sleep(delay)
    result = f"Результат задачи '{name}'"
    print(f"Задача '{name}' завершена.")
    future.set_result(result)  # Устанавливаем результат для Future объекта


async def main():
    # Создаем объект Future
    future = asyncio.Future()

    # Запускаем корутину, передаем Future объект в функцию
    await simulate_long_running_task("Задача1", 3, future)

    # Получаем результат выполнения задачи
    result = future.result()
    print(f"Результат Future: {result}")

# в Jupyter Notebook можно использовать await, в обычном скрипте нужно использовать asyncio.run() для запуска корутины
await main()

Правила и особенности использования `await`:
- `await` - это ключевое слово, которое используется для переключения между awaitable объектами
- `await` нельзя использовать с не awaitable объектами
- `await` нельзя использовать вне корутин
- `await` блокирует выполнение корутины
- `await` возвращает результат
- корутина должна содержать `await`, чтобы реализовать преимущества asyncio

Цикл событий - работает в одном потоке и занимается организацией задач: он отмечает, когда какая задача должна начаться, и запускает её в нужный момент. Каждая корутина, запущенная в цикле событий, общается "сообщениями" под капотом, передавая информацию о своём статусе для эффективного управления ею. Когда ваша программа включает в себя множество асинхронных операций, цикл событий может управлять их выполнением, переключаясь между задачами в оптимальном порядке.

![event_loop.png](img/event_loop.png)

Event loop обрабатывает события следующим образом:
- Сбор событий: Цикл событий отслеживает и собирает асинхронные события и задачи, которые нужно выполнить. Это включает в себя операции ввода/вывода, таймеры и запланированные асинхронные задачи. События и задачи могут поступать из различных источников и добавляются в очередь на обработку.

- Определение и распределение задач: Когда событие готово к обработке, event loop определяет соответствующую асинхронную задачу или корутину для его обработки. Приоритеты событий обычно определяются автоматически системой.

- Выполнение задач: Event loop запускает задачу и управляет её выполнением. Если задача асинхронна, она может быть приостановлена (await) для ожидания других событий (например, ответа от сервера) без блокировки цикла событий. Таким образом, event loop может переключаться между задачами, эффективно распределяя время процессора.

- Ожидание и завершение событий: После инициации задачи event loop продолжает отслеживать и обрабатывать другие входящие события. Он циклически возвращается к шагу сбора событий, пока есть активные или ожидающие задачи.

- Обработка ошибок: Если во время выполнения задачи возникает ошибка, она должна быть обработана внутри самой задачи или через механизмы обработки исключений asyncio. Цикл событий продолжит работу, если исключение будет корректно обработано. Необработанные исключения могут прервать выполнение задачи, но не обязательно останавливают сам цикл событий.

- Перезапуск цикла: Цикл событий продолжает работать до тех пор, пока не будет остановлен явно, либо до тех пор, пока не завершатся все задачи. Это позволяет приложению оставаться реактивным и обрабатывать события в реальном времени.

asyncio.run() — функция создаёт цикл событий и запускает указанную корутину в нём.

- `asyncio.run()` нельзя вызвать из корутины. Это приведет к ошибке, так как функцию нельзя вызывать, когда другой цикл событий asyncio уже работает в том же потоке.
- `asyncio.run()` может быть использован только с корутинами. Если вы попытаетесь передать обычную функцию вместо корутины, вы получите ошибку ValueError.
- Если в процессе выполнения функции main() произошла ошибка, то asyncio.run(main()) гарантирует, что все остальные асинхронные операции будут отменены, и затем пробросит исключение в вызывающий код. Для того, чтобы обработать исключение, необходимо добавить блок try/except на потенциально опасном участке кода.

asyncio.sleep() — временно приостанавливает исполнение текущей корутины, позволяя циклу событий (event loop) обрабатывать другие корутины или задачи.

Если вам нужно запустить задачи, добавленные в цикл событий, но не нужно явно ожидать результат выполнения задачи, можно использовать await asyncio.sleep(0). В этот момент произойдет переключение контекста и начнется выполнение задач. await asyncio.sleep(0) также можно использовать внутри корутин для возможности смены контекста. 

In [ ]:
# Чтобы добавить возможность смены контекста без дополнительного ожидания можно добавить
# await asyncio.sleep(0), и переключение контекста будет происходить после каждой итерации.
# Таким образом счетчики будут работать конкурентно.

import asyncio

async def count():
    for i in range(3, 0, -1):
        print(f"Обратный отсчет: {i}")
        await asyncio.sleep(0)

async def main():
    await asyncio.gather(count(), count())

# asyncio.run(main())
await main()

asyncio.create_task() — функция оборачивает coro в объект Task, т.е. она будет запланирована для выполнения в цикле событий

При создании задачи в цикле событий с помощью метода asyncio.create_task(my_coroutine()) корутина my_coroutine() оборачивается в объект Task и планируется ее выполнение. Выполняться задача начнет при первом переключении контекста (await).

In [ ]:
# Пример завершения главной задачи до завершения корутины
import asyncio

async def my_coroutine(seconds):
    print(f'Задача my_coroutine() запущена')
    await asyncio.sleep(seconds)
    print(f'Задача my_coroutine() завершена')

async def main():
    print(f'Задача main() запущена')
    task1 = asyncio.create_task(my_coroutine(3), name= "TestTask")
    print(f'Задача main() завершена')
    # await task1 # если раскомментировать эту строку, то main() будет ждать завершения my_coroutine()

# asyncio.run(main())
await main()

asyncio.ensure_future() —  это устаревшая функция, которая используется для запуска корутины в асинхронном режиме.

In [ ]:
import asyncio

async def my_coroutine(i):
    print(f"Начинается корутина {i}")
    await asyncio.sleep(1)
    print(f"Завершается корутина {i}")

async def main():
    tasks = [ asyncio.ensure_future(my_coroutine(i)) for i in range(5) ]
    await asyncio.gather(*tasks)


# asyncio.run(main())
await(main())

`asyncio.gather()` — функция позволяет группировать awaitable-объекты для их одновременного выполнения. Предназначена для конкурентного запуска awaitable объектов, переданных в нее, как последовательность *aws. Если все ожидания завершены успешно, результатом будет агрегированный список возвращаемых значений. Порядок значений результатов соответствует порядку ожиданий в aws.

Функция, возвращает объект класса asyncio.tasks._GatheringFuture (специфический future-объект, агрегирующий результаты переданных корутин или future).
Если какой-либо ожидаемый объект в aws является корутиной, он автоматически будет запланирован как задача (Task)


In [ ]:
# Пример использования asyncio.gather() для запуска нескольких корутин и получения их результатов
import asyncio
import time

async def coro(num):
    await asyncio.sleep(1/num)
    return f"Задача_{num} была выполнена за {time.perf_counter() - start:.4f} секунды"

async def main():
    future = asyncio.gather(*[coro(i) for i in range(1, 6)])
    print(type(future))
    results = await future
    print(results)

start = time.perf_counter()
# asyncio.run(main())
await(main())

`asyncio.wait_for(aw, timeout)` — функция ожидает завершения aw(awaitable объект) и, если выполнение не завершено в течение timeout секунд, переданный awaitable объект отменяется и вызывается TimeoutError.

In [ ]:
# Пример использования asyncio.wait_for() для установки таймаута в 3 секунды на выполнение корутины за 5 секунд
import asyncio

async def my_coro():
    await asyncio.sleep(5)
    return 'Привет мир!'

async def main():
    try:
        await asyncio.wait_for(my_coro(), timeout=3)
    except asyncio.TimeoutError:
        print("Корутина my_coro() не завершилась в течение 3 секунд и была прервана!")

# asyncio.run(main())
await(main())

`asyncio.wait(aws, *, timeout=None, return_when=asyncio.ALL_COMPLETED)` — функция используется для конкурентного запуска экземпляров Future и Task из итерируемого aws. Она приостанавливает выполнение текущей корутины до тех пор, пока не закончится таймаут или пока не наступит условие, указанное в параметре return_when, но это не влияет на выполнение других корутин или основной программы

Функция возвращает два множества Tasks/Futures: (done, pending):
- done: Множество задач, которые были выполнены или отменены.
- pending: Множество задач, которые ещё не были выполнены.

`return_when` может принимать одну из констант:
- `FIRST_COMPLETED` - возвращает управление (т.е., "разблокирует" выполнение текущей корутины), как только любая из ожидаемых Task/Future будет выполнена или отменена.
- `FIRST_EXCEPTION` - возвращает управление, как только любая из ожидаемых задач завершится с исключением 
- `ALL_COMPLETED` - значение по умолчанию, возвращает управление только после того, как все ожидаемые задачи будут выполнены или отменены

Функция не вызывает исключение `asyncio.TimeoutError` при наступлении таймаута. Задачи, которые не были выполнены к этому моменту, просто возвращаются во множестве pending и продолжают свое выполнение в фоновом режиме.

In [ ]:
# Пример использования asyncio.wait() с установленным таймаутом: управление вернется при завершении всех задач или при достижении таймаута

import asyncio

async def my_coro(delay):
    """Асинхронная задача, которая просто ждет указанное время."""
    task_name = asyncio.current_task().get_name()
    print(f'{task_name} началась, будет выполняться {delay} секунд(ы).')
    await asyncio.sleep(delay)
    print(f'{task_name} завершена.')
    return f'{task_name}: результат'

async def main():
    tasks = [asyncio.create_task(my_coro(i), name=f'Задача_{i}') for i in range(1, 4)]
    print("Запуск задач...")
    done, pending = await asyncio.wait(tasks, timeout=2.1)

    # Проверяем, какие задачи выполнены, а какие еще в ожидании
    print(f"\nРезультаты завершенных задач: {[d.result() for d in done]}")
    if pending:
        print(f"Ожидающие задачи: {[x.get_name() for x in pending]}")
        # Опционально: Можно отменить оставшиеся задачи, если это необходимо
        print(f"Отмена задач...")
        for task in pending:
            print(f"{task.get_name()}: Отмена ")
            task.cancel()
    else:
        print(f"Все задачи завершены.")

# asyncio.run(main())
await(main())

`asyncio.as_completed(aws, *, timeout=None)` - позволяет получать результаты асинхронных задач по мере их завершения, а не в порядке запуска. Функция возвращает итератор, который выдаёт futures по мере их завершения

In [ ]:
# Пример использования asyncio.as_completed() для обработки результатов задач по мере их завершения

import asyncio
import random

async def task(num):
    await asyncio.sleep(delay := random.random())
    return f'Task {num} completed, {delay=:.3f}'

async def main():
    tasks = [asyncio.create_task(task(i)) for i in range(5)]

    # as_completed() возвращает итератор корутин, который генерирует задачи по мере их завершения, позволяя обрабатывать результаты сразу после готовности
    for completed_task in asyncio.as_completed(tasks):
        # completed_task - объект корутины, создаваемый функцией as_completed(), возвращающий результат завершенной задачи.
        result = await completed_task
        print(result)

# asyncio.run(main())
await(main())

Согласно документации к модулю asyncio в большинстве случаев рекомендуется использовать более высокоуровневый `asyncio.run()`, но для взаимодействия с некоторыми библиотеками и фреймворками понадобится создавать цикл событий и обращаться к его методам. 

- `loop = asyncio.new_event_loop()` - создание нового цикла событий
- `asyncio.set_event_loop(loop)` - запуск созданного цикла событий (в однопоточном приложении этот шаг необязателен)
- `loop.run_until_complete(coro())` - запуск корутины из созданного цикла событий
- `loop.close()` - по окончании работы с циклом событий нужно закрыть его вручную

Особенности работы:
- Можно создать несколько разных циклов событий в рамках одного приложения
- В одном потоке может быть только один цикл событий в любой момент времени
- Цикл событий надо закрывать вручную


In [ ]:
import asyncio

async def main():
    print("Hello, World!")


loop = asyncio.new_event_loop()
# asyncio.set_event_loop(loop)

loop.run_until_complete(main())
loop.close()

In [ ]:
import asyncio
import threading


# Корутина, которая будет выполнена в каждом из независимых потоков.
async def task(loop, num):
    # Получаем id потока в котором выполняется задача.
    thread_id = threading.current_thread().ident
    # Получаем текущий цикл событий.
    e_loop_id = asyncio.get_running_loop()
    print(
        f"Задача task_{num} запущена в потоке id: {thread_id}, "
        f"текущий event loop id: {id(e_loop_id)}, "
        f"переданный event loop id: {id(loop)}")
    await asyncio.sleep(num)
    print(f"Задача task_{num} завершена!")


# Функция для запуска асинхронного цикла событий в отдельном потоке.
def start_loop(loop, coro, num):
    # Установка переданного цикла событий в качестве текущего для этого потока.
    asyncio.set_event_loop(loop)
    # Запуск задачи в установленном цикле событий, ожидание ее результата.
    loop.run_until_complete(coro(loop, num))
    # Закрываем цикл событий
    loop.close()


# Основная асинхронная функция, управляющая созданием циклов событий и потоков.
async def main():

    print(f'main()\nТекущий поток id: {threading.current_thread().ident}\n'
          f'Текущий цикл событий id: {id(asyncio.get_running_loop())}\n{"-" * 50}')
    loop1 = asyncio.new_event_loop()  # Создание нового цикла событий.
    print(f'Создан новый цикл событий id: {id(loop1)}')
    loop2 = asyncio.new_event_loop()  # Создание еще одного нового цикла событий.
    print(f'Создан новый цикл событий id: {id(loop2)}\n{"-" * 50}')

    # Создание и запуск независимых потоков для каждого цикла событий и корутины.
    thread1 = threading.Thread(target=start_loop, args=(loop1, task, 1,))
    thread2 = threading.Thread(target=start_loop, args=(loop2, task, 2))

    # Запуск потоков.
    thread1.start()
    thread2.start()
s
    # Ожидание завершения потоков.
    thread1.join()
    thread2.join()


asyncio.run(main())